In [7]:
%pip install -q --no-cache-dir \
    transformers==4.53.3 \
    safetensors==0.5.3 \
    trl==0.19.1 \
    peft==0.17.1 \
    accelerate==1.8.1 \
    bitsandbytes==0.46.1 \
    datasets==3.6.0 \
    scikit-learn==1.7.2 \
    sentencepiece==0.2.0

In [8]:

import numpy as np
import pandas as pd
import torch
import os
import random
from trl import *
from peft import *
import zipfile
from pathlib import Path
from transformers import *
from scipy.sparse import hstack
from datasets import Dataset

seed = 42
model_id = "Qwen/Qwen3-4B-Instruct-2507"

MAX_LENGTH = 1024
MAX_PROMPT_LENGTH = 256
MAX_COMPLETION_LENGTH = 768

out = Path("/content/outputs")
sft_d = out / "sft"
dp_d = out / "dpo"
rwm_d = out / "rwd_m"
dpo_qd = out / "dpo_q"
simp_d = out / "simpo"

os.environ["PYTHONHASHSEED"] = str(seed)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
set_seed(seed)

out.mkdir(parents=True, exist_ok=True)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report

In [10]:
from pathlib import Path
import json
import zipfile

archive_path = Path(
    "/content/ml-olympiad-red-task-c1005bf0-8695-451a-9616-87c8687dce27.zip"
)

with zipfile.ZipFile(archive_path) as archive:
    archive.extractall("/content")

data_dir = Path("/content/ml-olympiad-red-task/data")


def read_jsonl(filename):
    with open(data_dir / filename, encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


kid_adult = read_jsonl("kid_adult.jsonl")
good_bad = read_jsonl("good_bad.jsonl")
public_style = read_jsonl("public_test_style.jsonl")
public_quality = read_jsonl("public_test_quality.jsonl")


In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


def user(text):
    return [{"role": "user", "content": text}]


def assistant(text):
    return [{"role": "assistant", "content": text}]


sft_da = Dataset.from_list([
    {
        "prompt": user(row["prompt"]),
        "completion": assistant(row["kid"]),
    }
    for row in kid_adult
])

sty_da = Dataset.from_list([
    {
        "prompt": user(row["prompt"]),
        "chosen": assistant(row["kid"]),
        "rejected": assistant(row["adult"]),
    }
    for row in kid_adult
])

re_da = Dataset.from_list([
    {
        "chosen": [
            {"role": "user", "content": row["instruction"]},
            {"role": "assistant", "content": row["chosen"]},
        ],
        "rejected": [
            {"role": "user", "content": row["instruction"]},
            {"role": "assistant", "content": row["rejected"]},
        ],
    }
    for row in good_bad
])

qu_da = Dataset.from_list([
    {
        "prompt": user(row["instruction"]),
        "chosen": assistant(row["chosen"]),
        "rejected": assistant(row["rejected"]),
    }
    for row in good_bad
])

print(sft_da)
print(sty_da)
print(re_da)
print(qu_da)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Dataset({
    features: ['prompt', 'completion'],
    num_rows: 1489
})
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 1489
})
Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 2226
})
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 2226
})


In [16]:
import joblib

style_f = joblib.load(
    "/content/ml-olympiad-red-task/metrics/style_clf.pkl"
)
style_clf = style_file["clf"]
style_vecs = style_file["vecs"]

simp_cl = int(
    np.where(style_clf.classes_ == 1)[0][0]
)


def p_simp(texts):
    ft = hstack([
        vectorizer.transform(texts)
        for vectorizer in style_vecs
    ])

    prb = style_clf.predict_proba(ft)

    return float(
        prb[:, simp_cl].mean()
    )


print(
    "kid:",
    p_simp([row["kid"] for row in kid_adult])
)

print(
    "adult:",
    p_simp([row["adult"] for row in kid_adult])
)

kid: 0.9744881038704017
adult: 0.02487847940706435
